# llama-roofline on a CPU runtime

This notebook measures one machine end to end with nothing to install by hand:
the memory-bandwidth ceiling, then llama.cpp decode on one small model, then the
`diagnose` verdict (is token generation limited by memory bandwidth, and how close to
the wall is it). At the end it prints a filled-in body for a
[results gallery](https://github.com/manunicholasjacob/llama-roofline/issues/new?template=results-gallery.md)
issue, which you can paste if you want to share the result. Nothing is uploaded by the
notebook itself.

It runs on Google Colab's default CPU runtime and on any Linux x86-64 machine with
Python 3.9 or later. A GPU is not used: the question here is the CPU and system RAM.
Expect about 10 minutes on a 2-vCPU Colab runtime, most of it the benchmark.

Shared cloud machines are noisy neighbours. The tool keeps every repetition and flags a
run whose spread suggests contention; if the report says the ceiling was unstable, run
the last two cells again.

In [ ]:
%pip install -q "llama-roofline>=0.2.3"
import sys
LR = f'"{sys.executable}" -m llama_roofline'  # the CLI, from this kernel's own environment
!{LR} --version

## 1. llama-bench

`llama-roofline` drives llama.cpp's own `llama-bench`. This cell fetches the official
prebuilt Ubuntu x64 CPU build from the llama.cpp releases page. `LLAMA_CPP_BUILD` pins
the build the notebook was tested with; set it to `"latest"` to take the newest one.

In [ ]:
import json, os, pathlib, platform, tarfile, urllib.request

LLAMA_CPP_BUILD = "b11612"  # or "latest"
assert platform.system() == "Linux" and platform.machine() in ("x86_64", "AMD64"),     "this cell fetches the Linux x86-64 build; elsewhere, pass --llama-bench to your own build"

def release_asset(build):
    api = "https://api.github.com/repos/ggml-org/llama.cpp/releases"
    url = f"{api}?per_page=20" if build == "latest" else f"{api}/tags/{build}"
    with urllib.request.urlopen(url, timeout=60) as r:
        data = json.load(r)
    for rel in (data if isinstance(data, list) else [data]):
        for a in rel.get("assets", []):
            if a["name"].endswith("-bin-ubuntu-x64.tar.gz"):
                return rel["tag_name"], a["browser_download_url"]
    raise RuntimeError("no ubuntu-x64 CPU asset found for " + build)

tag, url = release_asset(LLAMA_CPP_BUILD)
root = pathlib.Path("llama.cpp-" + tag)
if not root.exists():
    tgz, _ = urllib.request.urlretrieve(url, root.name + ".tar.gz")
    with tarfile.open(tgz) as t:
        try:
            t.extractall(root, filter="data")
        except TypeError:  # Python without extraction filters
            t.extractall(root)
bench = next(p for p in root.rglob("llama-bench") if p.is_file())
bench.chmod(0o755)
os.environ["LLAMA_BENCH"] = str(bench.resolve())
print(tag, "->", os.environ["LLAMA_BENCH"])
!"$LLAMA_BENCH" --help | head -3

## 2. One small model

Qwen2.5-0.5B-Instruct at Q4_K_M, from Qwen's own GGUF repository (about 490 MB).
`inspect` reads only the file's header over HTTP range requests, so you can see what
the file contains, and the bytes decode will read per token, before downloading it.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct-GGUF/qwen2.5-0.5b-instruct-q4_k_m.gguf"
!{LR} inspect hf:{MODEL}

In [ ]:
owner, repo, fname = MODEL.split("/", 2)
model_path = pathlib.Path("models") / fname
model_path.parent.mkdir(exist_ok=True)
if not model_path.exists():
    urllib.request.urlretrieve(
        f"https://huggingface.co/{owner}/{repo}/resolve/main/{fname}", model_path)
print(model_path, round(model_path.stat().st_size / 1e6), "MB")

## 3. Diagnose

Measures the bandwidth ceiling with read-only kernels, benchmarks decode across a few
thread counts, and states the verdict. `--quick` (one repetition, fewer thread
settings) roughly halves the time and makes the numbers noisier.

In [ ]:
!{LR} diagnose {model_path} --out llama-roofline-out

## 4. Share it (optional)

The cell below prints an issue body for the results gallery: the machine description
and `diagnosis.md`, which is the shareable report. Copy it, open
[a new results-gallery issue](https://github.com/manunicholasjacob/llama-roofline/issues/new?template=results-gallery.md)
and paste. Read it first: it names the CPU and RAM size of the machine you ran on.
The last section of the template, "Did it get anything wrong?", is the useful part, and
only you can fill it in.

In [ ]:
def first(path, key):
    try:
        for line in open(path, encoding="utf-8", errors="replace"):
            if line.startswith(key):
                return line.split(":", 1)[1].strip() if ":" in line else line.split("=", 1)[1].strip().strip('"')
    except OSError:
        pass
    return "unknown"

cpu = first("/proc/cpuinfo", "model name")
mem_gb = int(first("/proc/meminfo", "MemTotal").split()[0]) / 1024 / 1024
os_name = first("/etc/os-release", "PRETTY_NAME=")
where = "Google Colab CPU runtime" if "COLAB_RELEASE_TAG" in os.environ else "Linux"
report = pathlib.Path("llama-roofline-out/diagnosis.md").read_text(encoding="utf-8")

body = f"""## Machine

{cpu}, {os.cpu_count()} logical CPUs, {mem_gb:.1f} GB RAM, {os_name} ({where}).
llama.cpp {tag} prebuilt ubuntu-x64 CPU build; model {fname}.

## Report

```
{report.rstrip()}
```

## Did it get anything wrong?

"""
print("Title: [results] " + cpu + (" (Colab)" if where.startswith("Google") else ""))
print()
print(body)